In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader

import torch
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset

import torch.optim as optim
from annoy import AnnoyIndex
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import mlflow
import random
from mlflow.models.signature import infer_signature

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_colwidth', None)


In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [4]:
files = ['product_brand_embedding.npy', 'product_bullet_point_embedding.npy', 'product_color_embedding.npy', 'product_title_embedding.npy', 'product_description_embedding.npy', 'query_embedding.npy']

data = []
for file in files:
    data.append(np.load(f'../data/new_embeddings/{file}'))

In [5]:
embedding = np.concat((data[0],data[1],data[2],data[3],data[4],data[5]),axis=1)

In [6]:
labels = pd.read_csv(f'../data/data.csv')['binary_label'].values

In [7]:
embedding.shape

(99909, 4608)

In [8]:
labels.shape

(99909,)

In [9]:
x_train, x_test, y_train, y_test = train_test_split(embedding, labels, test_size=0.2, random_state=42, shuffle=True)

In [10]:
x_train = torch.from_numpy(x_train)

x_test = torch.from_numpy(x_test)

y_train = torch.from_numpy(y_train)

y_test = torch.from_numpy(y_test)

In [11]:
query_embeddings = x_train[:,3840:]
product_embeddings = x_train[:,:3840]

In [12]:
query_embeddings.shape[1]

768

In [13]:
class TwoTowerModel(nn.Module):
    def __init__(self, input_dim, embedding_dim):
        super(TwoTowerModel, self).__init__()
        self.query_tower = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Linear(256, 32)
        )
        self.product_tower = nn.Sequential(
            nn.Linear(3840, 256),
            nn.ReLU(),
            nn.Linear(256, 32)
        )
    
    def forward(self, query_input, product_input):
        query_emb = self.query_tower(query_input)
        product_emb = self.product_tower(product_input)
        
        # Normalize embeddings
        query_emb = nn.functional.normalize(query_emb, p=2, dim=1)
        product_emb = nn.functional.normalize(product_emb, p=2, dim=1)
        
        return query_emb, product_emb
    
    def compute_similarity(self, query_emb, product_emb):
        return torch.matmul(query_emb, product_emb.T)  # Dot product similarity


In [14]:
7968/32

249.0

In [15]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, query_emb, pos_product_emb, neg_product_emb):
        pos_dist = torch.norm(query_emb - pos_product_emb, p=2, dim=1)
        neg_dist = torch.norm(query_emb - neg_product_emb, p=2, dim=1)
        # print(pos_dist.shape, neg_dist.shape)
        loss = torch.mean(torch.relu(pos_dist - neg_dist + self.margin))  # Triplet Loss
        # print(loss)
        return loss

# Initialize model and optimizer
input_dim = query_embeddings.shape[1] # 768
embedding_dim = 3840
model = TwoTowerModel(input_dim, embedding_dim)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = ContrastiveLoss()

# Training loop
num_epochs = 100
batch_size = 32

for epoch in range(num_epochs):
    model.train()
    
    for i in range(0, 7968, batch_size):
        query_batch = query_embeddings[i:i+batch_size]
        pos_product_batch = product_embeddings[i:i+batch_size]


        # Negative sampling: shuffle product embeddings
        neg_indices = torch.randint(0, len(product_embeddings), (batch_size,))
        neg_product_batch = product_embeddings[neg_indices]

        # Forward pass
        query_emb, pos_product_emb = model(query_batch, pos_product_batch)
        _, neg_product_emb = model(query_batch, neg_product_batch)

        # print(query_emb.shape, pos_product_emb.shape, neg_product_emb.shape)

        # Compute loss
        loss = criterion(query_emb, pos_product_emb, neg_product_emb)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch+1) % 10 == 0:
        # print(f'Epoch [{epoch+1}/{epochs}], Train Accuracy: {train_accuracy:.2f}%, Test Accuracy: {test_accuracy:.2f}%')
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")


Epoch 10/100, Loss: 0.1388
Epoch 20/100, Loss: 0.0730
Epoch 30/100, Loss: 0.0813
Epoch 40/100, Loss: 0.0618
Epoch 50/100, Loss: 0.1489
Epoch 60/100, Loss: 0.0527
Epoch 70/100, Loss: 0.1059
Epoch 80/100, Loss: 0.0116
Epoch 90/100, Loss: 0.0821
Epoch 100/100, Loss: 0.0281


In [16]:
embedding = torch.from_numpy(embedding)

In [17]:
import faiss

model.eval()
with torch.no_grad():
    _, product_embeddings_faiss = model(embedding[:,3840:], embedding[:,:3840])

In [18]:
product_embeddings_faiss_np = product_embeddings_faiss.numpy()

In [19]:
index = faiss.IndexFlatL2(32)
index.add(product_embeddings_faiss_np)  # Add product embeddings


In [20]:
# Function to retrieve top-k products for a query
def retrieve_products(query, product, top_k=5):
    query_emb, _ = model(query, product)  # Encode query
    query_emb_np = query_emb.detach().numpy()
    
    _, indices = index.search(query_emb_np, top_k)  # FAISS search
    return indices

In [21]:
df = pd.read_csv('../data/data.csv')

In [22]:
from sentence_transformers import SentenceTransformer

emb_model = SentenceTransformer('Alibaba-NLP/gte-multilingual-base',trust_remote_code=True)


Some weights of the model checkpoint at Alibaba-NLP/gte-multilingual-base were not used when initializing NewModel: {'classifier.weight', 'classifier.bias'}
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
def get_embeddings(text):
    embeddings = emb_model.encode(text, normalize_embeddings=True, show_progress_bar=True)
    return embeddings.squeeze()


In [24]:
emb_example = torch.from_numpy(get_embeddings(df.iloc[100].to_list()[1:7]))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [25]:
emb_example = emb_example.reshape((1,4608))

In [26]:
new_query = emb_example[:,3840:]
product = emb_example[:,:3840]

In [27]:
recommended_products = retrieve_products(new_query, product, top_k=5)
print("Recommended Product Indices:", recommended_products)


Recommended Product Indices: [[32658 34959 63361 59849 81865]]


In [28]:
recommended_products[0]

array([32658, 34959, 63361, 59849, 81865])

In [29]:
df.iloc[100]

product_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [30]:
df.iloc[recommended_products[0]]

,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,query,esci_label,split,binary_label
32658,B0818FCD15,brubaker wine bottle holder wedding couple table top metal sculpture with greeting card,an unforgettable wedding gift for every bridal couple bride and groom are slightly tilted forward and hold their hands as if they are about to kiss he wears a tail with a top hat and she wears a corsage dress with a lace skirt and a long lace veil in the middle is the holder for the bottle which is decorated in front with a large heart in which a small heart is hanging the wedding couple will delight you with its modern look made of welded metal parts and is a great eyecatcher in any living area it can be placed anywhere in the room and is beautiful from all sides give away the bottle stand with a bottle of wine liquor and other spirits a bottle of wine is a popular souvenir at weddings everyone is happy about a great bottle of wine thanks to the large diameter of the holders 335 inches this bottle stand easily holds standard wine bottles 254 fl oz as well as many other bottles just check the diameter of your bottle each bottle holder is designed by metal sculptors the bridal couple are made of pig iron which is galvanised at the end of the manufacturing process to give it a shiny finish the manufacturing process is done by hand from cutting the iron sheets bending the tubes and moulds to soldering the finished parts such as screws and nuts therefore each of our sculptures is an absolute unique piece with greeting card in vino veritas the bottle is not included in the deliverydimensions width 77 inches 195 cmheight 106 inches 27 cmdepth 48 inches 123 cmweight 720 g,an unforgettable gift for wedding couples which will inspire you with its many details\nevery bridal couple is looking forward to a great wine present your wine present with this wine accessory and your gift will become unique\nthis wine holder can also be a fabulously beautiful centerpiece on your own wedding table\nwine bottles liquor or oil bottles thanks to the large diameter of the holders 335 inches this bottle stand easily holds standard wine bottles 254 fl oz as well as many other bottles please check the diameter size\nwith greeting card in vino veritas the bottle is not included dimensions width 77 inches height 106 inches depth 48 inches,brubaker,silver,wine holder wedding,E,train,1
34959,B07BMQ3NCQ,key bottle openers assorted vintage skeleton keys wedding party favors pack of 70 silver,package include 70 pieces vintage key bottle openers assorted antique silver skeleton keys 10 pieces each style 7 styles quality design look and feel each is made of sturdy quality metal alloy and has an antiqued color finish to give them a beautiful vintage look the keys measure between 275 35 inches long,the key bottle openers are made of metal alloy they measure 275 to 35 long every key is well made unique and beautiful\nsilver color vintage style key shaped bottle opener favors used to be bottle opener thank you gifts a key to your seat on the wedding or party\nperfect gift on all kinds of wedding styles party favors\nuseful meaningful gift for your guests friends family the key to happiness is love\nthese keys are a delightful gift or addition for a greeting card thank you card wedding party themed birthday party mini treasure toy gifts,xonor,silver,liquor opener,E,test,1
63361,B07J3Y3GNH,pamtier mens stainless steel 2 pack beer bar tool creative versatile finger bottle opener ring size 8,quality warranty 1 nickel free hypoallergenic property 2 100 top quality workmanship and details you worth possessing 3 30 days quality warranty ring size 1 it is based on american ring size standard 2 please carefully check ring size you need before purchasing it is not resizable 3 jewelry stores always have the tool to check accurate ring size,vintage handmade well polished finish\nexcellent quality comfort fit and good price\nsimply place your hand 